# INIT


In [ ]:
%load_ext autoreload
%autoreload 2

from datetime import datetime
import plotly.io as pio
import urllib3
import warnings
import random
from pathlib import Path
import pickle

from prometheus_api_client import PrometheusConnect

from data_source.prometheus import *
from data_source.guidellm import (
    discover_runs,
    extract_time_ranges,
    load_runs,
    load_multi_instance_run,
    compute_latency_percentiles,
    compute_error_summary,
    compute_throughput_summary,
    compute_gating_verdicts as guidellm_gating_verdicts,
    compute_error_breakdown,
    compute_non_successful_timing,
    compute_error_timeline,
)
from transform.sampling import *
from plotting.load_signal_static import *
from plotting.violin_plots import *
from plotting.candlestick import *
from plotting.combine import *

random.seed(42)

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
warnings.simplefilter("ignore", FutureWarning)

# Configuration: $ oc port-forward -n openshift-monitoring svc/thanos-querier 9091:9091
PROMETHEUS_URL = "https://localhost:9091"
# oc create token bastion-daemon -n debugging --duration 240h
BEARER_TOKEN = "BEARER_TOKEN"
MODEL_NAME = "Qwen/Qwen3-32B"
NAMESPACE = "experiment-01"
VARIANT_NAME = "llm-d-inference-scheduling-llm-d-modelservice-decode"
GPUS_PER_REPLICA = 1

OUTPUT_FOLDER = Path("_out/flow-control-01")
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

QUERY = False
STORE = False # Will store queried data to avoid re-querying (needs QUERY=True first) - Ignored when LOAD=True, QUERY=False
LOAD = True





In [ ]:
# -- GuideLLM result files (primary, gateway-level data) --
# Run `./fetch-results.sh` first, then auto-discover from each PVC:
GUIDELLM_RESULTS = (
    discover_runs(OUTPUT_FOLDER / "benchmarks")
)
# Or explicit: [(["/path/to/inst0.json", "/path/to/inst1.json"], "label"), ...]

# -- TIME_RANGES for Prometheus queries --
# Auto-derived from GuideLLM JSON timestamps (with 60s padding for rate() warm-up).
# The labels match between both configs automatically.
if GUIDELLM_RESULTS:
    TIME_RANGES = extract_time_ranges(GUIDELLM_RESULTS)


In [ ]:
import pandas as pd
import numpy as np
from data_source.guidellm import load_report

rows = []

for json_paths, experiment_label in GUIDELLM_RESULTS:
    for json_path in json_paths:
        # Determine stream from path (high-priority or low-priority)
        if 'high-priority' in str(json_path):
            stream = 'high-priority'
        elif 'low-priority' in str(json_path):
            stream = 'low-priority'
        else:
            continue
        
        # Load the JSON report
        report = load_report(json_path)
        
        # Iterate over benchmarks (each benchmark is a different concurrency level)
        for bm in report.get('benchmarks', []):
            # Get concurrency level from strategy
            concurrency = bm.get('config', {}).get('strategy', {}).get('max_concurrency')
            
            # Get request metrics
            reqs = bm.get('requests', {})
            successful = reqs.get('successful', [])
            
            # Calculate latency percentiles (P50, P90, P99) from successful requests
            ttft = [r.get('time_to_first_token_ms') for r in successful if r.get('time_to_first_token_ms') is not None]
            tpot = [r.get('time_per_output_token_ms') for r in successful if r.get('time_per_output_token_ms') is not None]

            if ttft:
                p50_ttft = np.percentile(ttft, 50)
                p90_ttft = np.percentile(ttft, 90)
                p99_ttft = np.percentile(ttft, 99)
            else:
                p50_ttft = p90_ttft = p99_ttft = np.nan

            if tpot:
                p50_tpot = np.percentile(tpot, 50)
                p90_tpot = np.percentile(tpot, 90)
                p99_tpot = np.percentile(tpot, 99)
            else:
                p50_tpot = p90_tpot = p99_tpot = np.nan
            
            # Calculate error rate
            sm = bm.get('scheduler_metrics', {})
            rm = sm.get('requests_made', {})
            total = rm.get('total', 0)
            errored = rm.get('errored', 0)
            error_rate = (errored / total * 100) if total > 0 else 0.0
            
            # Calculate throughput (requests per second)
            duration = sm.get('measure_end_time', 0) - sm.get('measure_start_time', 0)
            successful_count = rm.get('successful', 0)
            throughput = (successful_count / duration) if duration > 0 else 0.0
            
            rows.append({
                'Experiment': experiment_label,
                'Stream': stream,
                'Concurrency': concurrency,
                'TTFT_P50': p50_ttft,
                'TTFT_P90': p90_ttft,
                'TTFT_P99': p99_ttft,
                'TPOT_P50': p50_tpot,
                'TPOT_P90': p90_tpot,
                'TPOT_P99': p99_tpot,
                'Error Rate': error_rate,
                'Throughput': throughput
            })

df = pd.DataFrame(rows)
df

In [ ]:
# Define colors for each experiment
experiments = df['Experiment'].unique()
import plotly.express as px
colors = px.colors.qualitative.Vivid
pio.templates.default = "plotly_white"
# Define symbols for streams
symbol_map = {
    'high-priority': 'triangle-up',
    'low-priority': 'triangle-down'
}

def create_metric_chart(df, metric_col, title, yaxis_title):
    """Create a scatter plot for a given metric vs concurrency."""
    fig = go.Figure()
    
    for i, experiment in enumerate(sorted(df['Experiment'].unique())):
        for stream in df['Stream'].unique():
            # Filter data for this experiment and stream
            mask = (df['Experiment'] == experiment) & (df['Stream'] == stream)
            data = df[mask].sort_values('Concurrency')
            
            if len(data) > 0:
                fig.add_trace(go.Scatter(
                    x=data['Concurrency'],
                    y=data[metric_col],
                    mode='lines+markers',
                    name=f'{experiment} - {stream}',
                    marker=dict(
                        symbol=symbol_map.get(stream, 'circle'),
                        size=10,
                        color=colors[i]
                    ),
                    line=dict(
                        color=colors[i],
                        dash='solid' if stream == 'high-priority' else 'dash'
                    )
                ))
    
    fig.update_layout(
        title=title,
        xaxis_title='Concurrency',
        yaxis_title=yaxis_title,
        hovermode='closest',
        showlegend=True
    )
    
    return fig

# Create charts for each metric
fig_ttft_p50 = create_metric_chart(df, 'TTFT_P50', 'TTFT P50 vs Concurrency', 'Latency P50 (ms)')
fig_ttft_p90 = create_metric_chart(df, 'TTFT_P90', 'TTFT P90 vs Concurrency', 'Latency P90 (ms)')
fig_ttft_p99 = create_metric_chart(df, 'TTFT_P99', 'TTFT P99 vs Concurrency', 'Latency P99 (ms)')
fig_tpot_p50 = create_metric_chart(df, 'TPOT_P50', 'TPOT P50 vs Concurrency', 'Time per Output Token P50 (ms)')
fig_tpot_p90 = create_metric_chart(df, 'TPOT_P90', 'TPOT P90 vs Concurrency', 'Time per Output Token P90 (ms)')
fig_tpot_p99 = create_metric_chart(df, 'TPOT_P99', 'TPOT P99 vs Concurrency', 'Time per Output Token P99 (ms)')

fig_throughput = create_metric_chart(df, 'Throughput', 'Throughput vs Concurrency', 'Throughput (req/s)')

# Display the charts
fig_ttft_p50.show()
fig_ttft_p90.show()
fig_ttft_p99.show()
fig_tpot_p50.show()
fig_tpot_p90.show()
fig_tpot_p99.show()
fig_throughput.show()

In [ ]:

if QUERY:
    prom = PrometheusConnect(
        url=PROMETHEUS_URL,
        disable_ssl=True,
        headers={"Authorization": f"Bearer {BEARER_TOKEN}"},
    )

if QUERY:
    queue_size_aggregated = custom_query_range_by_run(prom, TIME_RANGES, 'sum(vllm:num_requests_waiting{{model_name="{m}",namespace="{ns}"}})'.format(m=MODEL_NAME, ns=NAMESPACE), "1m", samples_generator_flat)
    queue_size_flow_control = custom_query_range_by_run(prom, TIME_RANGES, 'sum(inference_extension_flow_control_queue_size{{model_name="{m}",namespace="{ns}"}})'.format(m=MODEL_NAME, ns=NAMESPACE), "1m", samples_generator_flat)
    if STORE:
        queue_size_aggregated.to_parquet(OUTPUT_FOLDER / f"queue_size_aggregated.parquet")
        queue_size_flow_control.to_parquet(OUTPUT_FOLDER / f"queue_size_flow_control.parquet")

if LOAD:
    queue_size_aggregated = pd.read_parquet(OUTPUT_FOLDER / f"queue_size_aggregated.parquet")
    queue_size_flow_control = pd.read_parquet(OUTPUT_FOLDER / f"queue_size_flow_control.parquet")



In [ ]:
queue_size_violin = violin_plot_by_run(
    df=queue_size_aggregated,
    title="Queued Requests",
    yscale=1,
    xtitle="Scenario",
    yaxes_config=dict(
        title="Number of Queued Requests",
    )
)

queue_size_violin.show()